# Data preprocessing for machine learning



## 1. Setup and the split-first rule
- `X`: input features, shaped `(samples, features)`; `y`: target labels or values.
- **Train** learns parameters; **validation** guides choices; **test** is used once at the end.
- `fit` learns statistics. `transform` applies them. Use `fit_transform` only on training data.
- A fixed random seed makes this lesson reproducible.

**Exercise:** Which split should learn a missing-value median? Which split should help choose a model?


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
from IPython.display import display

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler, MinMaxScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.inspection import permutation_importance
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, mean_absolute_error

SEED = 42

def split_data(X, y, classification=False):
    """60% train, 20% validation, 20% test."""
    X_dev, X_test, y_dev, y_test = train_test_split(
        X, y, test_size=0.2, random_state=SEED,
        stratify=y if classification else None,
    )
    X_train, X_val, y_train, y_val = train_test_split(
        X_dev, y_dev, test_size=0.25, random_state=SEED,
        stratify=y_dev if classification else None,
    )
    return X_train, X_val, X_test, y_train, y_val, y_test


## 2. Imputation — fill missing values
**Data:** Titanic toy dataset from the original imputation notebook.

Use mean or median for numeric features; median is less sensitive to outliers. Use the most frequent value (mode) for categories. Filling values changes the distribution and does not recover the true missing values.


In [ ]:
toy_path = kagglehub.dataset_download("aman2626786/titanic-toy-dataset")
toy = pd.read_csv(Path(toy_path) / "titanic_toy.csv")
toy_X = toy[["Age", "Fare", "Family"]]
toy_y = toy["Survived"]
toy_train, toy_val, toy_test,toy_y_train, toy_y_val, toy_y_test = split_data(toy_X, toy_y, classification=True)

display(toy_train.head())
display(toy_train.isna().sum().rename("missing training values"))


In [ ]:
toy[["Age", "Fare", "Family"]]

In [ ]:
toy_imputer = ColumnTransformer([
    ("age_median", SimpleImputer(strategy="median"), ["Age"]),
    ("fare_mean", SimpleImputer(strategy="mean"), ["Fare"]),
    ("family_median", SimpleImputer(strategy="median"), ["Family"]),
], verbose_feature_names_out=False)

toy_train_filled = pd.DataFrame(
    toy_imputer.fit_transform(toy_train),
    columns=toy_imputer.get_feature_names_out(), index=toy_train.index,
)
toy_val_filled = toy_imputer.transform(toy_val)
toy_test_filled = toy_imputer.transform(toy_test)

display(toy_train_filled.head())
display(pd.DataFrame({
    "before variance": toy_train.var(),
    "after variance": toy_train_filled.var(),
}))


## 3. Encoding — turn categories into numbers
**Data:** Titanic, loaded directly through `seaborn`. This dataset will also be our complete pipeline example.

- **Nominal:** no meaningful order → one-hot encoding (e.g. embarkation port).
- **Ordinal:** meaningful order → specify it explicitly (e.g. low < medium < high).
- Impute categories before encoding.


In [ ]:
titanic = sns.load_dataset("titanic")
numeric_cols = ["age", "fare", "sibsp", "parch"]
categorical_cols = ["sex", "embarked", "pclass"]

X = titanic[numeric_cols + categorical_cols]
y = titanic["survived"]
X_train, X_val, X_test, y_train, y_val, y_test = split_data(
    X, y, classification=True,
)
display(X_train.head())
print("Train / validation / test:", X_train.shape, X_val.shape, X_test.shape)


In [ ]:
category_demo = Pipeline([
    ("impute", SimpleImputer(strategy="most_frequent")),
    ("encode", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
    #handle_unknown: what to do when an unknown categorical data comes while testing?,
    #sparse_output=False: maintains full matrix of 1s and 0s, if True then saves memory by only storing 1s with its position
])
encoded_train = category_demo.fit_transform(X_train[categorical_cols])
encoded_val = category_demo.transform(X_val[categorical_cols])
display(pd.DataFrame(
    encoded_train, columns=category_demo.get_feature_names_out(categorical_cols),
    index=X_train.index,
).head())



## 4. Standardization — put numeric features on comparable scales
**Data:** Social Network Ads from the original standardization notebook.

`StandardScaler`: **z = (x − training mean) / training standard deviation**.

Training columns have mean ≈ 0 and standard deviation ≈ 1 (except constant columns). Validation/test statistics need not match. Scaling helps gradient-based learning, including neural networks, and distance-based methods. It does not make data normally distributed or remove outliers.


In [ ]:
ads_path = kagglehub.dataset_download("rakeshrau/social-network-ads")
ads = pd.read_csv(Path(ads_path) / "Social_Network_Ads.csv")
(ads_train, ads_val, ads_test,
 ads_y_train, ads_y_val, ads_y_test) = split_data(
    ads[["Age", "EstimatedSalary"]], ads["Purchased"], classification=True,
)
ads_scaler = StandardScaler()
ads_train_scaled = pd.DataFrame(
    ads_scaler.fit_transform(ads_train), columns=ads_train.columns, index=ads_train.index,
)
ads_val_scaled = ads_scaler.transform(ads_val)
ads_test_scaled = ads_scaler.transform(ads_test)

display(pd.DataFrame({
    "original mean": ads_train.mean(),
    "original std": ads_train.std(),
    "scaled mean": ads_train_scaled.mean(),
    "scaled std": ads_train_scaled.std(),
}).round(3))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3))
axes[0].scatter(ads_train["Age"], ads_train["EstimatedSalary"], s=12)
axes[0].set(title="Before scaling", xlabel="Age", ylabel="Estimated salary")
axes[1].scatter(ads_train_scaled["Age"], ads_train_scaled["EstimatedSalary"], s=12)
axes[1].set(title="After scaling", xlabel="Standardized age", ylabel="Standardized salary")
plt.tight_layout()
plt.show()


**Exercise:** Fit a `MinMaxScaler` on `ads_train` and transform `ads_val`. Compare the ranges with `StandardScaler`. Can a validation value fall outside [0, 1]? Why?




## 5. Assemble one reusable pipeline
Return to Titanic. `Pipeline` chains steps; `ColumnTransformer` applies different steps to different columns.

```text
Numeric columns     → median imputation → standardization ┐
Categorical columns → mode imputation   → one-hot encoding ┴→ classifier
```

Encoding and scaling run in parallel branches. Keep the target separate; do not standardize classification labels. A pipeline also keeps preprocessing inside training folds when used with cross-validation.


In [ ]:
numeric_pipeline = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
])
categorical_pipeline = Pipeline([
    ("impute", SimpleImputer(strategy="most_frequent")),
    ("encode", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])
preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_cols),
    ("categorical", categorical_pipeline, categorical_cols),
])
classifier = Pipeline([
    ("preprocess", preprocessor),
    ("model", LogisticRegression(max_iter=1000, random_state=SEED)),
])
classifier.fit(X_train, y_train)
print("Validation accuracy:", accuracy_score(y_val, classifier.predict(X_val)))


## 6. Feature importance — inspect a fitted model
**Data:** diamonds from the original feature-importance notebook. Predict `price` using numeric features.

Permutation importance shuffles one validation feature and measures the drop in a model's score.


In [ ]:
diamonds = sns.load_dataset("diamonds")
diamond_cols = ["carat", "depth", "table", "x", "y", "z"]
diamonds

In [ ]:
diamonds = sns.load_dataset("diamonds")
diamond_cols = ["carat", "depth", "table", "x", "y", "z"]
(diamond_train, diamond_val, diamond_test,
 diamond_y_train, diamond_y_val, diamond_y_test) = split_data(
    diamonds[diamond_cols], diamonds["price"],
)
diamond_model = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
    ("model", LinearRegression()),
])
diamond_model.fit(diamond_train, diamond_y_train)
print("Validation MAE:", mean_absolute_error(
    diamond_y_val, diamond_model.predict(diamond_val),
))

importance = permutation_importance(
    diamond_model, diamond_val, diamond_y_val,
    scoring="neg_mean_absolute_error", n_repeats=5, random_state=SEED,
)
importance_table = pd.DataFrame({
    "feature": diamond_cols,
    "mean increase in MAE": importance.importances_mean,
    "std": importance.importances_std,
}).sort_values("mean increase in MAE")
display(importance_table)
importance_table.plot.barh(
    x="feature", y="mean increase in MAE", xerr="std", legend=False, figsize=(7, 3),
)
plt.xlabel("Increase in validation MAE after shuffling")
plt.tight_layout()
plt.show()


## 7. PCA — compress numeric features
**Data:** the inline height/weight/age example from the original PCA notebook. We use it only to demonstrate compression.

PCA creates new, uncorrelated training components that capture variance. It does not select original columns or use labels. Scale first, then fit PCA on training data only. Explained variance is not prediction accuracy; useful low-variance information can be lost.


In [ ]:
measurements = pd.DataFrame({
    "Height": [170, 165, 180, 175, 160, 172, 168, 177, 162, 158],
    "Weight": [65, 59, 75, 68, 55, 70, 62, 74, 58, 54],
    "Age": [30, 25, 35, 28, 22, 32, 27, 33, 24, 21],
})
pca_train, pca_holdout = train_test_split(
    measurements, test_size=0.3, random_state=SEED,
)
pca_pipeline = Pipeline([
    ("scale", StandardScaler()),
    ("pca", PCA(n_components=2)),
])
train_components = pca_pipeline.fit_transform(pca_train)
holdout_components = pca_pipeline.transform(pca_holdout)
pca = pca_pipeline.named_steps["pca"]
print("Shape before / after:", pca_train.shape, train_components.shape)
print("Explained variance ratio:", pca.explained_variance_ratio_.round(3))
print("Total retained variance:", pca.explained_variance_ratio_.sum().round(3))

display(pd.DataFrame(
    pca.components_, columns=measurements.columns, index=["PC1", "PC2"],
))
plt.scatter(train_components[:, 0], train_components[:, 1], label="Train")
plt.scatter(holdout_components[:, 0], holdout_components[:, 1], marker="x", label="Holdout")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.legend()
plt.tight_layout()
plt.show()


## 8. Final evaluation and practice
After choosing the Titanic pipeline using validation results, evaluate once on the test set.


In [ ]:
print("Final Titanic test accuracy:", accuracy_score(y_test, classifier.predict(X_test)))
